# Model Invocation and Error Handling

Real calls fail: Bedrock throttles you, a model is not enabled, the network drops, a tool raises an exception. This notebook shows how to handle each case, from plain `try/except` to the v1 **middleware**.

**Use case:** the calculator agent from notebook 04.

| Problem | Tool |
|---|---|
| Bad credentials, model id, throttling | `try/except ClientError` |
| Slow or flaky network | boto3 `Config` (timeouts, retries) |
| Temporary Bedrock errors | `ModelRetryMiddleware` |
| Model unavailable | `ModelFallbackMiddleware` |
| A tool raises an exception | `ToolRetryMiddleware`, `ToolErrorMiddleware` |
| Runaway loops (cost) | `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware` |


## Setup

In [ ]:
%%capture
%pip install --upgrade langgraph langchain langchain-aws langchain-community
%pip install --upgrade python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load AWS credentials from .env (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION)
# boto3 / langchain-aws pick them up from the environment automatically.
load_dotenv()


## Timeouts and retries on the model

`config` is a boto3 `Config`. The `standard` retry mode already retries throttling and 5xx errors with backoff.

In [ ]:
from botocore.config import Config
from langchain_aws import ChatBedrockConverse

model = ChatBedrockConverse(
    model="amazon.nova-lite-v1:0",
    temperature=0,
    config=Config(
        connect_timeout=10,
        read_timeout=30,
        retries={"max_attempts": 3, "mode": "standard"},
    ),
)


## Part 1: `try/except` around a model call

Bedrock errors arrive as `botocore.exceptions.ClientError`. The error code tells you what to do.

In [ ]:
from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError


def safe_invoke(llm, prompt: str) -> str:
    try:
        return llm.invoke(prompt).text
    except ClientError as e:
        code = e.response["Error"]["Code"]
        message = e.response["Error"]["Message"]
        if code == "ThrottlingException":
            return "Bedrock is busy. Try again in a moment."
        if code == "AccessDeniedException":
            return "No access to this model. Enable it in the Bedrock console (Model access)."
        if code in ("ValidationException", "ResourceNotFoundException"):
            return f"Bad request or model id ({code}): {message}"
        raise  # unknown error: do not hide it
    except NoCredentialsError:
        return "No AWS credentials found. Check your .env file."
    except BotoCoreError as e:
        return f"Network or client problem: {type(e).__name__}"


In [ ]:
print(safe_invoke(model, "What is 12 plus 8? Answer with just the number."))

bad_model = ChatBedrockConverse(model="amazon.nova-does-not-exist-v1:0")
print(safe_invoke(bad_model, "What is 12 plus 8?"))


## Part 2: retry the model call automatically

`ModelRetryMiddleware` retries failed model calls with exponential backoff. Middleware goes in `create_agent(middleware=[...])`.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

retry_agent = create_agent(
    model=model,
    system_prompt="You are a calculator assistant.",
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,        # up to 3 retries after the first attempt
            initial_delay=1.0,    # seconds
            backoff_factor=2.0,   # 1s, 2s, 4s ...
            on_failure="continue",  # after the last retry, return an error message instead of crashing
        )
    ],
)

retry_agent.invoke({"messages": "What is 12 plus 8?"})["messages"][-1].text


## Part 3: fall back to another model

If the first model fails, `ModelFallbackMiddleware` tries the next one. Here the primary model id is wrong on purpose.

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

fallback_agent = create_agent(
    model=bad_model,                                  # fails
    middleware=[ModelFallbackMiddleware(model)],      # works
)

fallback_agent.invoke({"messages": "What is 12 plus 8?"})["messages"][-1].text


## Part 4: when a tool fails

A tool is just Python code, so it can raise. This `flaky_add` fails twice, then works (like a calculator service that is briefly down).

`ToolRetryMiddleware` retries the tool call; `retry_on` limits which exceptions are retried.

In [ ]:
from langchain_core.tools import tool
from langchain.agents.middleware import ToolRetryMiddleware

calls = {"n": 0}


@tool
def flaky_add(a: float, b: float) -> float:
    """Add two numbers: a + b."""
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("calculator service unavailable")
    return a + b


tool_retry_agent = create_agent(
    model=model,
    tools=[flaky_add],
    system_prompt="Always use the flaky_add tool for addition.",
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            initial_delay=0.2,
            backoff_factor=1.5,
            jitter=False,
            retry_on=(ConnectionError,),
        )
    ],
)

result = tool_retry_agent.invoke({"messages": "What is 12 plus 8?"})
print("tool executions:", calls["n"])
print(result["messages"][-1].text)


### A tool that raises an error the model should see

Retrying does not help for `10 / 0`: the input is the problem. Better: send the error back to the model as a `ToolMessage` so it can explain it to the user. `ToolErrorMiddleware` does this for the exceptions you choose.

In [ ]:
@tool
def strict_divide(a: float, b: float) -> float:
    """Divide a by b: a / b."""
    if b == 0:
        raise ValueError("division by zero")
    return a / b


plain_agent = create_agent(model=model, tools=[strict_divide])

try:
    plain_agent.invoke({"messages": "What is 10 divided by 0?"})
    print("The agent handled it on its own.")
except Exception as e:
    print("Agent crashed:", type(e).__name__, "-", e)


In [ ]:
from langchain.agents.middleware import ToolErrorMiddleware


def on_error(exc: Exception, request) -> str | None:
    if isinstance(exc, ValueError):
        return f"{type(exc).__name__}: that calculation is not possible. Explain this to the user."
    return None  # anything else still stops the run


safe_agent = create_agent(
    model=model,
    tools=[strict_divide],
    middleware=[ToolErrorMiddleware(on_error)],
)

for msg in safe_agent.invoke({"messages": "What is 10 divided by 0?"})["messages"]:
    msg.pretty_print()


## Part 5: stop runaway loops

An agent that keeps calling the model or tools costs money. Set limits. With `exit_behavior="end"` the run stops politely instead of raising.

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware


@tool
def add(a: float, b: float) -> float:
    """Add two numbers: a + b."""
    return a + b


limited_agent = create_agent(
    model=model,
    tools=[add],
    system_prompt="Use the add tool, one call at a time.",
    middleware=[
        ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
        ToolCallLimitMiddleware(run_limit=2),
    ],
)

result = limited_agent.invoke(
    {"messages": "Add 1 and 2, then add 3 to that, then add 4, then add 5, then add 6."}
)
print(result["messages"][-1].text)


## Summary

* Always wrap raw model calls in `try/except ClientError` and branch on the error code.
* Set timeouts and retries on the model with a boto3 `Config`.
* Agents: add `ModelRetryMiddleware`, `ModelFallbackMiddleware`, `ToolRetryMiddleware` and `ToolErrorMiddleware` instead of writing loops yourself.
* Put call limits on every agent that can loop.
* Middleware order matters: they wrap each other in the order of the list.
